---
# STEP 08 FPI 지도 시각화

## 분석 목적

FPI 분석 결과를 공간적으로 시각화하여 창업자와 상권 컨설턴트가
경쟁 압력 수준을 지도 위에서 직관적으로 파악할 수 있는 실무 도구를 제공한다.

| 단계 | 내용 |
|---|---|
| STEP 8-1 | 전체 독립 브랜드 FPI 지도 (FPI 수치 연속형) |
| STEP 8-2 | 전체 독립 브랜드 FPI 구간 지도 (NP / LP / HP) |
| STEP 8-3 | 업종별 FPI 구간 지도 (Pizza / Mexican / Steakhouses / Sushi Bars / Burgers) |

**입력 데이터**
- `biz_indie_with_groups.csv`: 독립 브랜드 FPI 구간 + 위치 정보

**산출물**
- `fpi_map.html`: FPI 수치 연속형 지도
- `fpi_map_group.html`: FPI 구간별 지도
- `fpi_map_pizza.html`: Pizza 업종 FPI 지도
- `fpi_map_mexican.html`: Mexican 업종 FPI 지도
- `fpi_map_steakhouses.html`: Steakhouses 업종 FPI 지도
- `fpi_map_sushi_bars.html`: Sushi Bars 업종 FPI 지도
- `fpi_map_burgers.html`: Burgers 업종 FPI 지도

**활용 대상**

| 대상 | 활용 방안 |
|---|---|
| 창업자 | 상권 진입 전 경쟁 압력 수준 사전 파악 → 입지 선정 |
| 상권 컨설턴트 | 업종별 FPI 분포로 상권 분석 고도화 |
| 프랜차이즈 본사 | 신규 출점 시 주변 독립 브랜드 경쟁 환경 파악 |

---
## STEP 8-1. 전체 독립 브랜드 FPI 지도 (FPI 수치 연속형)

Las Vegas 독립 브랜드의 FPI 분포를 지도에 시각화한다.
창업자가 상권 진입 전 경쟁 압력 수준을 사전 파악할 수 있는 실무 활용 도구다.

- **색깔**: FPI 수치 (초록 → 낮은 압력 / 빨강 → 높은 압력)
- **크기**: 리뷰수 (매장 규모)
- **hover**: 업체명, 별점, FPI, 구간

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

PATH_to_data = "../results"
PATH_to_save = "../results"

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [14]:
indie_groups = pd.read_csv(f"{PATH_to_data}/biz_indie_with_groups.csv")
indie_groups = indie_groups[indie_groups['is_open'] == 1].copy()

print(f"독립 브랜드: {len(indie_groups):,}개")

# 공통 전처리
indie_map = indie_groups.copy()
indie_map['fpi_group'] = indie_map['fpi_group'].fillna('NP')
group_label = {'NP': '무풍지대 (NP)', 'LP': '저압력 (LP)', 'HP': '고압력 (HP)'}
indie_map['구간'] = indie_map['fpi_group'].map(group_label)

# FPI 구간별 색깔 지도 (NP/LP/HP 구분)
color_map = {
    '무풍지대 (NP)': '#4a90e2',
    '저압력 (LP)':   '#f5a623',
    '고압력 (HP)':   '#e65100'
}

독립 브랜드: 3,017개


In [10]:
import plotly.express as px

# FPI 구간 한글 라벨
group_label = {'NP': '무풍지대 (NP)', 'LP': '저압력 (LP)', 'HP': '고압력 (HP)'}

fig = px.scatter_mapbox(
    indie_map,
    lat='latitude',
    lon='longitude',
    color='fpi_300m',
    color_continuous_scale='RdYlGn_r',
    size='review_count',
    size_max=20,
    hover_name='name',
    hover_data={
        'stars': True,
        'fpi_300m': ':.3f',
        '구간': True,
        'review_count': True,
        'neighborhood': True,
        'latitude': False,
        'longitude': False
    },
    zoom=11,
    height=700,
    title='Las Vegas 독립 브랜드 FPI 분포 지도 (fpi_300m 기준)'
)
fig.update_layout(
    mapbox_style='open-street-map',
    margin={'r': 0, 't': 50, 'l': 0, 'b': 0},
    coloraxis_colorbar=dict(
        title='FPI (경쟁압력)',
        tickvals=[0, 2, 4, 6, 8],
        ticktext=['0 (무풍)', '2', '4', '6', '8↑ (고압력)']
    )
)
fig.write_html(f"{PATH_to_save}/fpi_map.html")
print("저장 완료: fpi_map.html")
fig.show()

저장 완료: fpi_map.html


---
## STEP 8-2. 전체 독립 브랜드 FPI 구간 지도 (NP / LP / HP)

STEP 9-1의 연속형 FPI 수치를 NP/LP/HP 3개 구간으로 범주화하여 시각화한다.
연속형 지도보다 구간 간 경계가 명확해 상권별 경쟁압력 수준을 직관적으로 파악할 수 있다.

- **색깔**: FPI 구간 (파랑=무풍지대 NP / 주황=저압력 LP / 빨강=고압력 HP)
- **크기**: 리뷰수 (매장 규모)
- **hover**: 업체명, 별점, FPI 수치, 구간, 상권명

In [ ]:
fig2 = px.scatter_mapbox(
    indie_map,
    lat='latitude',
    lon='longitude',
    color='구간',
    color_discrete_map=color_map,
    size='review_count',
    size_max=20,
    hover_name='name',
    hover_data={
        'stars': True,
        'fpi_300m': ':.3f',
        '구간': True,
        'review_count': True,
        'neighborhood': True,
        'latitude': False,
        'longitude': False
    },
    zoom=11,
    height=700,
    title='Las Vegas 독립 브랜드 FPI 구간 분포 지도'
)
fig2.update_layout(
    mapbox_style='open-street-map',
    margin={'r': 0, 't': 50, 'l': 0, 'b': 0}
)
fig2.write_html(f"{PATH_to_save}/fpi_map_group.html")
print("저장 완료: fpi_map_group.html")
fig2.show()

저장 완료: fpi_map_group.html


---
## FPI 지도 결과 해석

### 공간적 패턴 관찰

**① Strip 중심부 = 프랜차이즈 경쟁 과밀 구역**

Paradise, The Strip 인근 주요 관광 동선을 따라 FPI가 압도적으로 높게 나타난다.
이는 단순히 프랜차이즈 수가 많다는 의미를 넘어, 접근성 높은 관광 상권에
체인 브랜드 선호 소비층이 집중되어 있음을 의미한다.

> 독립 브랜드 입장에서 Strip 상권은 매출 잠재력은 크지만
> 가격·브랜드·광고 경쟁 부담이 극심한 레드오션이다.
> 이 지역에서 생존하려면 차별화된 컨셉, 특정 문화/경험 제공,
> niche targeting 전략이 필수적이다.

---

**② 도심 외곽 = 저압력 분산형 구조**

Summerlin, Spring Valley 등 서쪽·남쪽 주거지역으로 갈수록
LP/NP 비중이 높아지며 프랜차이즈 공백 지역이 다수 존재한다.

> 외곽 주거 상권은 관광객보다 로컬 수요 중심으로,
> 임대료 부담이 낮고 단골 기반 운영이 가능하여
> 독립 브랜드 생존 가능성이 상대적으로 높다.
> 커뮤니티 기반 매장, 지역 특화 브랜드에 유리한 환경이다.

---

**③ 프랜차이즈 경쟁은 면(area)이 아닌 교통축(linear corridor) 중심**

구간화 지도에서 HP가 주요 대로변·교차로를 따라 선형으로 이어지는 패턴이 나타난다.
이는 프랜차이즈 확장이 도시 전체를 균일하게 커버하는 것이 아니라
접근성 높은 교통축 중심으로 집중되고 있음을 시사한다.

---

**④ 무풍지대(NP)는 완전히 사라지지 않는다**

프랜차이즈가 도시 전체를 균일하게 장악했다면 NP는 거의 존재하지 않아야 한다.
그러나 외곽은 물론 일부 중심부에도 NP가 곳곳에 존재한다.

> 이는 프랜차이즈 확장이 도시를 완전히 커버하지 못하며,
> 독립 브랜드에게 niche opportunity, 지역 특화 시장,
> 상권 공백이 여전히 존재한다는 시사점을 준다.

---

### 보고서용 결론 문장

> Las Vegas의 독립 브랜드는 관광 중심 상권에서 높은 프랜차이즈 경쟁압력(FPI)에
> 노출되어 있으며, 특히 Strip 축을 따라 경쟁이 집중되는 공간적 패턴이 나타났다.
> 반면 외곽 주거지역에서는 저압력 또는 무풍지대가 다수 확인되어,
> 독립 브랜드의 생존 가능성이 상대적으로 높은 지역적 틈새시장(niche market)이
> 존재함을 확인하였다. 경쟁압력은 단순 면적 기반이 아니라
> 주요 교통축과 상권 네트워크를 따라 선형적으로 형성되는 경향을 보였다.

---

### 입지 선정 가이드

| 상권 유형 | FPI 수준 | 특성 | 추천 전략 |
|---|---|---|---|
| Strip/Paradise | 고압력 (4↑) | 관광객 중심, 레드오션 | 경험형 컨셉, SNS 바이럴, 관광객 특화 |
| 주요 대로변 | 중간 (1~3) | 교통축 상권 | 정통 요리 전문성, 서비스 차별화 |
| 외곽 주거지 | 저압력~무풍 (0~1) | 로컬 수요 중심 | 단골 기반, 커뮤니티형, 에스닉 전문점 |

---
## STEP 8-3. 업종별 FPI 구간 지도

### 분석 설계 배경

Yelp 데이터의 categories 컬럼은 계층 구조 없이 세미콜론(;)으로 나열된 구조다.
예: `"Restaurants;Pizza;Italian"`

따라서 Restaurants의 하위 카테고리가 따로 존재하지 않으며,
Pizza, Steakhouses 등은 Restaurants와 동등한 태그로 붙어있다.

**FPI 계산은 Restaurants 전체 기준을 유지하는 이유:**
피자 독립 브랜드 입장에서 경쟁 압력은 피자 프랜차이즈만 받는 게 아니라
맥도날드, 서브웨이 등 주변 모든 프랜차이즈의 영향을 받는다.
따라서 FPI는 전체 Restaurants 기준으로 산출하고,
업종 필터링은 **어떤 독립 브랜드를 지도에 표시할지**만 결정한다.

**분석 대상 업종 (Restaurants 카테고리 포함 조건)**

| 업종 | 선정 이유 |
|---|---|
| Pizza | HP 구간 고유 키워드, 프랜차이즈 경쟁 뚜렷 |
| Mexican | NP/LP 구간 에스닉 전문점 대표 업종 |
| Steakhouses | LP 구간 정통 요리 대표, 수업 분석 업종 |
| Sushi Bars | 생존 브랜드 키워드(sushi, roll)와 직결 |
| Burgers | HP 구간 고유 키워드 1위, 프랜차이즈 직접 경쟁 |

In [12]:
cats = indie_groups['categories'].dropna()\
       .str.split(';').explode().str.strip()
print(cats.value_counts().head(10))

categories
Restaurants               3017
Food                       743
Nightlife                  546
Bars                       525
American (Traditional)     443
Mexican                    394
American (New)             333
Pizza                      322
Breakfast & Brunch         288
Sandwiches                 255
Name: count, dtype: int64


In [ ]:
import plotly.express as px

# 분석할 업종 5개
target_categories = {
    'Pizza': 'Pizza',
    'Mexican': 'Mexican',
    'Steakhouses': 'Steakhouses',
    'Sushi Bars': 'Sushi Bars',
    'Burgers': 'Burgers'
}

group_label = {'NP': '무풍지대 (NP)', 'LP': '저압력 (LP)', 'HP': '고압력 (HP)'}

for cat_name, cat_key in target_categories.items():
    # 해당 업종 필터링
    cat_df = indie_map[
        indie_map['categories'].str.contains('Restaurants', na=False) &
        indie_map['categories'].str.contains(cat_key, na=False)
    ].copy()

    print(f"{cat_name}: {len(cat_df)}개")

    fig = px.scatter_mapbox(
        cat_df,
        lat='latitude',
        lon='longitude',
        color='구간',
        color_discrete_map=color_map,
        size='review_count',
        size_max=20,
        hover_name='name',
        hover_data={
            'stars': True,
            'fpi_300m': ':.3f',
            '구간': True,
            'review_count': True,
            'neighborhood': True,
            'latitude': False,
            'longitude': False
        },
        zoom=11,
        height=700,
        title=f'Las Vegas 독립 브랜드 FPI 분포 — {cat_name} ({len(cat_df)}개)'
    )
    fig.update_layout(
        mapbox_style='open-street-map',
        margin={'r': 0, 't': 50, 'l': 0, 'b': 0}
    )
    filename = cat_name.lower().replace(' ', '_')
    fig.write_html(f"{PATH_to_save}/fpi_map_{filename}.html")
    print(f"  저장 완료: fpi_map_{filename}.html")
    fig.show()

Pizza: 322개
  저장 완료: fpi_map_pizza.html


Mexican: 395개
  저장 완료: fpi_map_mexican.html


Steakhouses: 146개
  저장 완료: fpi_map_steakhouses.html


Sushi Bars: 146개
  저장 완료: fpi_map_sushi_bars.html


Burgers: 180개
  저장 완료: fpi_map_burgers.html


## 업종별 경쟁 패턴 분석

업종별로 경쟁 압력(FPI) 분포에서 뚜렷한 차이가 나타났다.

---

### Pizza (499개)

HP(주황/빨강)가 Strip 중심축에 밀집되어 있으며, 외곽에도 LP/NP가 함께 분포한다.  
피자는 도미노, 파파존스와 같은 대형 프랜차이즈와 직접 경쟁하는 업종이기 때문에 Strip 주변에 HP가 집중되는 패턴이 나타난다.

---

### Burgers (276개)

HP가 Strip 축에 극단적으로 집중되어 있고 NP 비율이 매우 낮다.  
맥도날드, 버거킹 등 대형 프랜차이즈가 버거 시장을 강하게 장악하고 있어 독립 버거 브랜드의 생존 공간이 제한적인 구조를 보인다.  
5개 업종 중 경쟁 강도가 가장 높은 업종이다.

---

### Sushi Bars (255개)

다른 업종 대비 도시 전역에 비교적 고르게 분산되어 있다.  
NP/LP 비율이 상대적으로 높고 외곽 지역에도 다수 분포한다.  
스시는 프랜차이즈 경쟁이 약한 에스닉 전문 업종 특성상 독립 브랜드의 생존 가능성이 상대적으로 높은 환경으로 해석된다.

---

### Steakhouses (223개)

Strip 및 Paradise 축에 강하게 집중되어 있으며 HP 비율이 압도적으로 높다.  
카지노 호텔 내부의 고급 스테이크하우스 비중이 높은 업종 특성이 반영된 결과로, STEP 8에서 도출된 hotel, casino 중심 키워드와도 연결된다.

---

### Mexican (582개)

5개 업종 중 가장 넓은 공간적 분산을 보인다.  
NP(파랑)도 다양한 지역에 나타나며 외곽 주거지역까지 고르게 분포한다.  
멕시칸 음식은 프랜차이즈 경쟁 강도가 상대적으로 낮고 지역 기반 수요가 강한 업종이기 때문에 외곽 지역에서도 독립 브랜드의 생존 가능성이 높은 것으로 해석된다.

---

## 업종별 창업 난이도 요약

| 업종 | HP 집중도 | 독립 브랜드 생존 환경 |
|---|---|---|
| Burgers | 매우 높음 | 가장 어려움 |
| Steakhouses | 높음 | Strip 의존도 높음 |
| Pizza | 중간 | 외곽 기회 존재 |
| Sushi Bars | 낮음 | 비교적 유리 |
| Mexican | 가장 낮음 | 가장 유리 |